

---

PREPROCESSING DATA


---
1. Data Pangan: https://www.bi.go.id/hargapangan/TabelHarga/PasarTradisionalDaerah
2. Data Cuaca: https://power.larc.nasa.gov/data-access-viewer/
3. Data Kurs: https://finance.yahoo.com/quote/IDR=X/history/
4. FIX Dataset: https://drive.google.com/open?id=1NJsVHPmEj90ywkLVCmxPofgPWk6JAleM&usp=drive_copy

---




0. IMPORT LIBRARY

In [ ]:
import pandas as pd
import numpy as np
import sys
import holidays

1. PERBAIKAN DATA PANGAN (DATA WIDE -> LONG)

In [ ]:
print("Sedang memproses Data Pangan...")
try:
    # Baca Excel
    df_pangan = pd.read_excel('data_pangan.xlsx')

    # Buang kolom 'No' jika ada
    if 'No' in df_pangan.columns:
        df_pangan = df_pangan.drop(columns=['No'])

    # TRANSFORMASI DATA (MELT)
    # Asumsi kolom pertama adalah nama Komoditas, sisanya adalah Tanggal
    id_var = df_pangan.columns[0] # 'Komoditas (Rp)'
    df_pangan = df_pangan.melt(id_vars=[id_var], var_name='Date_Str', value_name='Harga_Beras')

    # Bersihkan Tanggal (Kadang format '01/ 01/ 2020')
    df_pangan['Date'] = pd.to_datetime(df_pangan['Date_Str'].astype(str).str.replace(' ', ''), format='%d/%m/%Y', errors='coerce')

    # Bersihkan Harga (Hapus 'Rp', koma, dan tanda strip '-')
    def clean_price(x):
        if isinstance(x, str):
            x = x.replace('Rp', '').replace('.', '').replace(',', '').strip()
            if x == '-': return np.nan
        return float(x)

    df_pangan['Harga_Beras'] = df_pangan['Harga_Beras'].apply(clean_price)

    # Ambil rata-rata jika ada data ganda di tanggal yang sama
    df_pangan = df_pangan.groupby('Date')['Harga_Beras'].mean().reset_index()

    print(f"✅ Data Pangan OK! ({len(df_pangan)} baris)")

except Exception as e:
    print(f"❌ Gagal baca Pangan: {e}")
    # Stop jika pangan gagal, karena ini data utama
    sys.exit()

Sedang memproses Data Pangan...
✅ Data Pangan OK! (1538 baris)


2. PERBAIKAN DATA CUACA (FIX HEADER & DOY)

In [ ]:
print("\nSedang memproses Data Cuaca...")
header_row = 0
with open('POWER_Point_Daily.csv', 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        # Cari baris yang mengandung YEAR dan (MO atau DOY)
        if 'YEAR' in line and ('MO' in line or 'DOY' in line):
            header_row = i
            print(f"   Header ditemukan di baris ke-{i}")
            break

# Baca CSV dengan skip baris yang benar
df_cuaca = pd.read_csv('POWER_Point_Daily.csv', skiprows=header_row)

# Cek apakah pakai DOY (Day of Year) atau MO/DY
if 'DOY' in df_cuaca.columns:
    # Konversi YEAR + DOY ke Date
    df_cuaca['Date'] = pd.to_datetime(df_cuaca['YEAR'] * 1000 + df_cuaca['DOY'], format='%Y%j')
else:
    # Konversi YEAR + MO + DY ke Date
    df_cuaca['Date'] = pd.to_datetime(df_cuaca[['YEAR', 'MO', 'DY']].astype(str).agg('-'.join, axis=1))

df_cuaca = df_cuaca[['Date', 'PRECTOTCORR', 'T2M']]
df_cuaca.columns = ['Date', 'Curah_Hujan', 'Suhu']
print(f"✅ Data Cuaca OK! ({len(df_cuaca)} baris)")


Sedang memproses Data Cuaca...
   Header ditemukan di baris ke-10
✅ Data Cuaca OK! (2153 baris)


3. DATA KURS

In [ ]:
print("\nSedang memproses Data Kurs...")
df_kurs = pd.read_csv('data_kurs_2020_2025.csv')
df_kurs['Date'] = pd.to_datetime(df_kurs['Date'])
# Ambil kolom ke-2 (Nilai Kurs)
df_kurs = df_kurs.iloc[:, [0, 1]]
df_kurs.columns = ['Date', 'Kurs_USD']


Sedang memproses Data Kurs...


4. MERGING (PENGGABUNGAN FINAL)

In [ ]:
print("\nMenggabungkan semua data...")
# Buat rentang tanggal lengkap (Master Timeline)
start_date = df_pangan['Date'].min()
end_date = df_pangan['Date'].max()
df_final = pd.DataFrame({'Date': pd.date_range(start=start_date, end=end_date)})

# Left Join ke Master Timeline
df_final = pd.merge(df_final, df_pangan, on='Date', how='left')
df_final = pd.merge(df_final, df_cuaca, on='Date', how='left')
df_final = pd.merge(df_final, df_kurs, on='Date', how='left')

# Tambah Fitur Libur
id_holidays = holidays.Indonesia()
df_final['Is_Holiday'] = df_final['Date'].apply(lambda x: 1 if x in id_holidays else 0)

# Isi Data Kosong (Interpolasi)
print("   Mengisi data kosong...")
df_final['Harga_Beras'] = df_final['Harga_Beras'].interpolate(method='linear')
df_final['Curah_Hujan'] = df_final['Curah_Hujan'].interpolate(method='linear')
df_final['Suhu'] = df_final['Suhu'].interpolate(method='linear')
df_final['Kurs_USD'] = df_final['Kurs_USD'].fillna(method='ffill').fillna(method='bfill')

# Final Check & Save
df_final = df_final.dropna() # Hapus sisa jika ada (biasanya di ujung awal/akhir)
df_final.to_csv('dataset_lengkap_lstm.csv', index=False)

print("\n🎉 SUKSES! File 'dataset_lengkap_lstm.csv' siap dipakai LSTM.")
print(df_final.head())


Menggabungkan semua data...
   Mengisi data kosong...

🎉 SUKSES! File 'dataset_lengkap_lstm.csv' siap dipakai LSTM.
        Date   Harga_Beras  Curah_Hujan   Suhu      Kurs_USD  Is_Holiday
1 2020-01-02  11466.666667         6.77  27.38  13689.231445           0
2 2020-01-03  11550.000000        51.20  27.25  13884.786133           0
3 2020-01-04  11527.777778         9.00  27.21  13884.786133           0
4 2020-01-05  11505.555556        10.65  27.10  13884.786133           0
5 2020-01-06  11483.333333         7.20  27.54  13934.316406           0


/tmp/ipython-input-3848263064.py:21: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_final['Kurs_USD'] = df_final['Kurs_USD'].fillna(method='ffill').fillna(method='bfill')
